In [2]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')
token=os.getenv("GITHUB_TOKEN")
if token:
    print("TOKEN LOADED SUCCESSFULLY")
    print(f"Token starts with: {token[:8]}...")
else:
    print("token not found")
    

TOKEN LOADED SUCCESSFULLY
Token starts with: ghp_xGE3...


In [8]:
from github import Github, Auth

auth = Auth.Token(token)
g = Github(auth=auth)

user = g.get_user()
print(f" Connected to GitHub as: {user.login}")
rate_limit = g.get_rate_limit()
print(f"Rate limit remaining: {g.rate_limiting[0]}/5000")

 Connected to GitHub as: Nishka2505
Rate limit remaining: 5000/5000


In [9]:
repo = g.get_repo("microsoft/vscode")
print(f"Repo: {repo.full_name}")
print(f"Stars: {repo.stargazers_count}")
print(f"Open PRs: {repo.get_pulls(state='open').totalCount}")

Repo: microsoft/vscode
Stars: 186277
Open PRs: 2138


In [11]:
import time
import pandas as pd
from tqdm import tqdm

REPOS = [
    "microsoft/vscode",
    "facebook/react",
    "kubernetes/kubernetes"
]

def is_bug_fix(pr):
    labels = [l.name.lower() for l in pr.labels]
    title = pr.title.lower()
    body = (pr.body or '').lower()
    keywords = ['bug', 'fix', 'crash', 'error', 'issue', 'defect', 'regression']
    return any(k in labels or k in title or k in body for k in keywords)

def collect_prs(repo_name, max_prs=300):
    print(f"\nCollecting from {repo_name}...")
    repo = g.get_repo(repo_name)
    records = []
    
    prs = repo.get_pulls(state='closed', sort='updated', direction='desc')
    
    for pr in prs:
        if len(records) >= max_prs:
            break
            
        if not pr.merged_at:
            continue
            
        try:
            reviews = list(pr.get_reviews())
            files = list(pr.get_files())
            
            record = {
                'pr_number'      : pr.number,
                'title'          : pr.title,
                'body'           : (pr.body or '')[:500],
                'additions'      : pr.additions,
                'deletions'      : pr.deletions,
                'changed_files'  : pr.changed_files,
                'commits'        : pr.commits,
                'comments'       : pr.comments,
                'review_comments': pr.review_comments,
                'num_reviewers'  : len(set(r.user.login for r in reviews if r.user)),
                'author'         : pr.user.login if pr.user else 'unknown',
                'reviewers'      : str([r.user.login for r in reviews if r.user]),
                'files_changed'  : str([f.filename for f in files]),
                'diff_patch'     : ' '.join([f.patch or '' for f in files])[:3000],
                'is_bug_fix'     : int(is_bug_fix(pr)),
                'created_at'     : pr.created_at,
                'merged_at'      : pr.merged_at,
                'repo'           : repo_name
            }
            records.append(record)
            
            # Print progress every 50 PRs
            if len(records) % 50 == 0:
                print(f"  Collected {len(records)}/{max_prs} PRs...")
                
            time.sleep(0.8)  # be respectful to API rate limits
            
        except Exception as e:
            print(f"  Skipped PR #{pr.number}: {e}")
            continue
    
    print(f"  Done — {len(records)} PRs from {repo_name}")
    return records

print("Function is defined ")

Function is defined 


In [12]:
all_records = []

for repo in REPOS:
    records = collect_prs(repo, max_prs=300)
    all_records.extend(records)
    print(f"Total so far: {len(all_records)} PRs")
    time.sleep(5)  # pause between repos

print(f"\n Collection complete — {len(all_records)} total PRs")


  Collected 50/300 PRs...
  Collected 100/300 PRs...
  Collected 150/300 PRs...
  Collected 200/300 PRs...
  Collected 250/300 PRs...
  Collected 300/300 PRs...
  Done — 300 PRs from microsoft/vscode
Total so far: 300 PRs



Following Github server redirection from /repos/facebook/react to /repositories/10270250


  Collected 50/300 PRs...
  Collected 100/300 PRs...
  Collected 150/300 PRs...
  Collected 200/300 PRs...
  Collected 250/300 PRs...
  Collected 300/300 PRs...
  Done — 300 PRs from facebook/react
Total so far: 600 PRs

  Collected 50/300 PRs...
  Collected 100/300 PRs...
  Collected 150/300 PRs...
  Collected 200/300 PRs...
  Collected 250/300 PRs...
  Collected 300/300 PRs...
  Done — 300 PRs from kubernetes/kubernetes
Total so far: 900 PRs

 Collection complete — 900 total PRs


In [13]:
df = pd.DataFrame(all_records)

# Save raw data
df.to_csv('../data/raw/pr_data.csv', index=False)

print(f" Saved to data/raw/pr_data.csv")
print(f"\nShape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nBug fix distribution:")
print(df['is_bug_fix'].value_counts())
print(f"\nBug fix rate: {df['is_bug_fix'].mean():.1%}")
print(f"\nPRs per repo:")
print(df['repo'].value_counts())
print(f"\nSample row:")
df.head(2)

 Saved to data/raw/pr_data.csv

Shape: (900, 18)

Columns: ['pr_number', 'title', 'body', 'additions', 'deletions', 'changed_files', 'commits', 'comments', 'review_comments', 'num_reviewers', 'author', 'reviewers', 'files_changed', 'diff_patch', 'is_bug_fix', 'created_at', 'merged_at', 'repo']

Bug fix distribution:
is_bug_fix
1    672
0    228
Name: count, dtype: int64

Bug fix rate: 74.7%

PRs per repo:
repo
microsoft/vscode         300
facebook/react           300
kubernetes/kubernetes    300
Name: count, dtype: int64

Sample row:


,pr_number,title,body,additions,deletions,changed_files,commits,comments,review_comments,num_reviewers,author,reviewers,files_changed,diff_patch,is_bug_fix,created_at,merged_at,repo
0,318992,fix(terminal): track ligatures addon config fo...,Store ILigatureOptions in _ligaturesAddonConfi...,10,6,1,3,4,0,2,Tyriar,"['anthonykim1', 'dmitrivMS']",['src/vs/workbench/contrib/terminal/browser/xt...,"@@ -139,7 +139,7 @@ export class XtermTerminal...",1,2026-05-29 15:47:32+00:00,2026-06-14 06:27:20+00:00,microsoft/vscode
1,321302,Browser: fix URLs incorrectly matching file://...,`Uri.parse()` defaults to adding a `file:` sch...,1,3,1,1,1,2,2,kycutler,"['copilot-pull-request-reviewer[bot]', 'dmitri...",['src/vs/workbench/contrib/browserView/electro...,"@@ -7,8 +7,6 @@ import { localize } from '../....",1,2026-06-14 04:08:49+00:00,2026-06-14 05:19:53+00:00,microsoft/vscode


In [14]:
print("=== DATA QUALITY CHECK ===")
print(f"\nMissing values:")
print(df.isnull().sum())

print(f"\nAverage PR size: {df['additions'].mean():.0f} lines added")
print(f"Average files changed: {df['changed_files'].mean():.1f}")
print(f"Unique authors: {df['author'].nunique()}")
print(f"\nDate range:")
print(f"  Earliest PR: {df['created_at'].min()}")
print(f"  Latest PR:   {df['created_at'].max()}")

print("\n  task 1 — data collected and saved")

=== DATA QUALITY CHECK ===

Missing values:
pr_number          0
title              0
body               0
additions          0
deletions          0
changed_files      0
commits            0
comments           0
review_comments    0
num_reviewers      0
author             0
reviewers          0
files_changed      0
diff_patch         0
is_bug_fix         0
created_at         0
merged_at          0
repo               0
dtype: int64

Average PR size: 423 lines added
Average files changed: 9.5
Unique authors: 227

Date range:
  Earliest PR: 2013-05-29 20:20:53+00:00
  Latest PR:   2026-06-14 04:08:49+00:00

  task 1 — data collected and saved
